# Dataset & split audit
Loads the frozen manifest, per-class counts, split sizes, and the sha256 integrity
checks. Run top-to-bottom; every assertion matches audio_dataset/scripts/verify_dataset.py.

In [ ]:
import csv, hashlib
from collections import Counter
from pathlib import Path

MANIFEST = Path('../audio_dataset/manifest_with_split.csv')
rows = list(csv.DictReader(MANIFEST.open()))
print('rows:', len(rows))
per_class = {}
splits = {}
for r in rows:
    per_class[r['class_label']] = per_class.get(r['class_label'], 0) + 1
    splits[r['dataset_split']] = splits.get(r['dataset_split'], 0) + 1
print('per class:', per_class)
print('splits:', splits)
assert len(per_class) == 10
assert all(v == 300 for v in per_class.values())
assert splits == {'train': 2100, 'val': 450, 'test': 450}
print('OK: 300 x 10 classes, 2100/450/450 split')

In [ ]:
# Class balance across splits (stratification evidence)
import matplotlib.pyplot as plt
matrix = {s: {} for s in ('train', 'val', 'test')}
for r in rows:
    matrix[r['dataset_split']][r['class_label']] = matrix[r['dataset_split']].get(r['class_label'], 0) + 1
classes = sorted(per_class)
fig, ax = plt.subplots(figsize=(10, 4))
import numpy as np
x = np.arange(len(classes))
w = 0.27
for i, s in enumerate(('train', 'val', 'test')):
    ax.bar(x + (i-1)*w, [matrix[s].get(c, 0) for c in classes], w, label=s)
ax.set_xticks(x); ax.set_xticklabels(classes, rotation=45, ha='right')
ax.legend(); ax.set_title('Stratified split: per class per split')
plt.tight_layout(); plt.show()

In [ ]:
# sha256 spot-check: 20 random rows hash correctly on disk
import hashlib, random
random.seed(7)
sample = random.sample(rows, 20)
bad = []
for r in sample:
    p = Path('../audio_dataset') / r['filename']
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    if digest != r['sha256']:
        bad.append(r['audio_id'])
assert not bad, bad
print('OK: 20/20 hashes match')